# 🌍 US Macroeconomic Dashboard
**Built by Ahmed Shareek**

This dashboard tracks four key US macroeconomic indicators in real time using data from the Federal Reserve Economic Data (FRED) API. It is designed to monitor the health of the US economy and flag early warning signals particularly yield curve inversions, which historically precede recessions.

---

In [ ]:
from fredapi import Fred

# Replace with your own API key (yours from earlier)
fred = Fred(api_key='YOUR_API_KEY_HERE') # Get your free API key at fred.stlouisfed.org

# Pull US GDP data
gdp = fred.get_series('GDP')

# Show the last 5 entries
print(gdp.tail())

2025-01-01    30042.113
2025-04-01    30485.729
2025-07-01    31098.027
2025-10-01    31422.526
2026-01-01    31865.721
dtype: float64


In [12]:
# Pull 10-Year and 2-Year Treasury yields
ten_year = fred.get_series('DGS10')
two_year = fred.get_series('DGS2')

# Calculate the spread (this is the key recession indicator)
spread = ten_year - two_year

# Show the most recent values
print(spread.tail())

2026-06-23    0.34
2026-06-24    0.30
2026-06-25    0.31
2026-06-26    0.31
2026-06-29    0.28
dtype: float64


## 1. Yield Curve Spread (10 Year minus 2 Year Treasury)
The yield curve spread measures the difference between long term and short term US government bond yields. Under normal conditions, long term bonds yield more than short term bonds. When this relationship inverts (spread goes negative), it has historically been one of the most reliable leading indicators of a recession. The red dashed line marks the inversion threshold.

In [13]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=spread.index, y=spread.values, name='10Y-2Y Spread'))

# Add a horizontal line at zero — anything below this line means inversion
fig.add_hline(y=0, line_dash='dash', line_color='red')

fig.update_layout(
    title='US Treasury Yield Curve Spread (10Y - 2Y)',
    xaxis_title='Date',
    yaxis_title='Spread (%)'
)

fig.show()

## 2. Inflation Rate (Year over Year CPI Change)
The Consumer Price Index (CPI) measures changes in the price of a basket of goods and services over time. This chart shows the year over year percentage change the "inflation rate" referenced in financial news and central bank policy decisions. Note the sharp rise in 2022 which prompted aggressive Federal Reserve rate hikes.

In [14]:
# Pull CPI (inflation) data
cpi = fred.get_series('CPIAUCSL')

# Calculate year-over-year % change — this is the actual "inflation rate" people talk about
inflation_rate = cpi.pct_change(periods=12) * 100

# Plot it
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=inflation_rate.index, y=inflation_rate.values, name='YoY Inflation %'))

fig2.update_layout(
    title='US Inflation Rate (Year-over-Year CPI Change)',
    xaxis_title='Date',
    yaxis_title='Inflation (%)'
)

fig2.show()

## 3. Unemployment Rate
The unemployment rate measures the percentage of the labour force actively seeking work but unable to find it. It is a lagging indicator it typically rises after a recession has already begun. The dramatic spike in 2020 reflects the COVID-19 economic shock and subsequent rapid recovery.

In [15]:
# Pull unemployment rate
unemployment = fred.get_series('UNRATE')

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=unemployment.index, y=unemployment.values, name='Unemployment Rate'))

fig3.update_layout(
    title='US Unemployment Rate',
    xaxis_title='Date',
    yaxis_title='Unemployment (%)'
)

fig3.show()

## 4. GDP Growth Rate (Year-over-Year)
Gross Domestic Product (GDP) measures the total value of goods and services produced in the US economy. This chart shows the year over year percentage change. Two consecutive quarters of negative growth is the technical definition of a recession. The red dashed line marks the zero growth threshold.

In [17]:
# Calculate GDP growth rate (year-over-year % change)
gdp_growth = gdp.pct_change(periods=4) * 100  # periods=4 because GDP is quarterly

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=gdp_growth.index, y=gdp_growth.values, name='GDP Growth Rate'))

# Add a zero line — below this means the economy is contracting
fig4.add_hline(y=0, line_dash='dash', line_color='red')

fig4.update_layout(
    title='US GDP Growth Rate (Year-over-Year)',
    xaxis_title='Date',
    yaxis_title='Growth (%)'
)

fig4.show()

In [16]:
# Check current yield curve status and flag recession risk
latest_spread = spread.dropna().iloc[-1]
latest_date = spread.dropna().index[-1]

print(f"Latest data point: {latest_date.strftime('%Y-%m-%d')}")
print(f"10Y-2Y Spread: {latest_spread:.2f}%")
print()

if latest_spread < 0:
    print("⚠️  RECESSION RISK ELEVATED — Yield curve is INVERTED")
    print("Short-term yields exceed long-term yields, historically a reliable recession signal.")
else:
    print("✅ Yield curve is NORMAL (not inverted)")
    print("No recession warning from this indicator at this time.")

Latest data point: 2026-06-29
10Y-2Y Spread: 0.28%

✅ Yield curve is NORMAL (not inverted)
No recession warning from this indicator at this time.
